In [ ]:
%pip install -Uq bitsandbytes==0.49.2 sentence-transformers==5.6.0 chromadb==1.5.9

In [ ]:
import os, gc, sys, shutil, subprocess, joblib, chromadb, logging
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import polars as pl

from pathlib import Path

from dotenv import load_dotenv
from kaggle_secrets import UserSecretsClient

from tqdm.auto import tqdm

from pacmap import PaCMAP
from hdbscan import HDBSCAN

import torch
from sentence_transformers import SentenceTransformer

from transformers import (
    BitsAndBytesConfig, AutoTokenizer, 
    AutoModelForCausalLM
)

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

In [ ]:
TRAIN_PATH = Path("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

CHROMA_DB_DATA = Path("/kaggle/input/datasets/spandanjit2005/knowledge-db")
CHROMA_DB_PATH = Path("knowledge-db")
CHROMA_DB_NAME = "knowledge_db"

if CHROMA_DB_PATH.exists():
    shutil.rmtree(CHROMA_DB_PATH)

shutil.copytree(CHROMA_DB_DATA, CHROMA_DB_PATH)

print("Copied files:\n")
for root, _, files in os.walk(CHROMA_DB_PATH):
    for f in files:
        p = os.path.join(root, f)
        print(f"{p}  ({os.path.getsize(p)} bytes)")

_verify_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
_verify_collection = _verify_client.get_collection(name=CHROMA_DB_NAME)

_count = _verify_collection.count()
assert _count > 0, "Collection is empty after copying from the Kaggle dataset"

_check = _verify_collection.get(limit=5, include=["embeddings"])
assert len(_check["ids"]) == 5, "Could not retrieve embeddings - vector segment may not have been copied"

_emb = np.array(_check["embeddings"][0], dtype=np.float32) # type: ignore
_sanity = _verify_collection.query(query_embeddings=[_emb.tolist()], n_results=1)
assert _sanity["ids"][0][0] == _check["ids"][0], \
    "Query did not return the vector's own nearest neighbor - index looks corrupted"

print(f"\nChroma DB copy verified: {_count} vectors present and searchable.")

del _verify_client, _verify_collection, _check, _emb, _sanity
gc.collect()

In [ ]:
# Configure HuggingFace API Key
load_dotenv()

# os.environ["HF_TOKEN"] = os.getenv("HF_READ_TOKEN") if os.getenv("HF_READ_TOKEN") else "" # type: ignore
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_READ_TOKEN")

# Configure logging levels to hide model-loading report
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

hf_logging.set_verbosity_error()

disable_progress_bar()

In [ ]:
train_data = pl.read_csv(TRAIN_PATH)
test_data = pl.read_csv(TEST_PATH)

model_config = {
    "embedder_model": "Qwen/Qwen3-Embedding-4B",
    "reranker_model": "Qwen/Qwen3-Reranker-8B",
    "generative_slm": "Qwen/Qwen2.5-14B-Instruct"
}

inter_path_config = {
    "working_dir": Path("/kaggle/working"),
    
    "train_retrieved": Path("train_retrieved.parquet"),
    "test_retrieved": Path("test_retrieved.parquet"),
    "train_ranked": Path("train_ranked.parquet"),
    "test_ranked": Path("test_ranked.parquet"),
    "train_final": Path("train_final.parquet"),
    "test_final": Path("test_final.parquet"),

    "submission": Path("submission.csv")
}

OPTION_COLS = ["A", "B", "C", "D", "E"]

SEED = 42

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
    # llm_int8_enable_fp32_cpu_offload=True
)

def reset_gpu() -> None:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

MCQ_EMBED_INSTRUCTION = (
    "Given a multiple-choice science question and its answer options, retrieve document "
    "chunks that contain the information needed to determine the correct answer"
)

MCQ_RERANK_INSTRUCTION = (
    "Given a multiple-choice question and its answer options, judge whether this "
    "chunk contains information that helps determine the correct answer"
)

GENERATIVE_SLM_INSTRUCTION = (
    "You are an expert scientist. Use the provided context to answer the multiple-choice "
    "question. Respond with only the letter of the single best option."
)

In [ ]:
train_data = train_data.with_columns(
    (
        pl.lit("Prompt : ") + pl.col("prompt")
        + pl.lit(" Options: ")
        + pl.concat_str(
            [pl.lit(f"{opt}) ") + pl.col(opt).fill_null(" ") for opt in OPTION_COLS],
            separator=" "
        )
    ).alias("mcq_query")
)

test_data = test_data.with_columns(
    (
        pl.lit("Prompt : ") + pl.col("prompt")
        + pl.lit(" Options: ")
        + pl.concat_str(
            [pl.lit(f"{opt}) ") + pl.col(opt).fill_null(" ") for opt in OPTION_COLS],
            separator=" "
        )
    ).alias("mcq_query")
)

In [ ]:
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH) 
collection = chroma_client.get_collection(name=CHROMA_DB_NAME)

In [ ]:
def retrieve_top_k(
    query_embeddings: np.ndarray | list[list[float]], 
    k: int = 10, 
    overfetch: int = 15, 
    batch_size: int = 50
) -> list[list[str]]:
    
    query_embeddings = np.asarray(query_embeddings)
    all_docs = []

    for start in tqdm(range(0, len(query_embeddings), batch_size), desc="Retrieving"):
        batch = query_embeddings[start:start + batch_size]
        results = collection.query(query_embeddings=batch.tolist(), n_results=k + overfetch)

        for docs in results["documents"]: # type: ignore
            seen = set()
            unique_docs = []
            
            for d in docs:
                if d not in seen:
                    seen.add(d)
                    unique_docs.append(d)
                    
                if len(unique_docs) == k:
                    break

            if len(unique_docs) < k:
                print(f"Warning: only {len(unique_docs)}/{k} unique chunks retrieved")

            all_docs.append(unique_docs)

    for i, docs in enumerate(all_docs):
        if len(docs) < k: print(f"Row {i} has {len(docs)} chunks, expected {k}")

    return all_docs

In [ ]:
embedder = SentenceTransformer(
    model_config["embedder_model"],
    model_kwargs={
        "trust_remote_code": True,
        "dtype": torch.float16,
        "device_map": "cuda:0"
    },
    processor_kwargs={"padding_side": "left"},
    prompts={"mcq_query": f"Instruct: {MCQ_EMBED_INSTRUCTION}\nQuery: "},
    default_prompt_name="mcq_query"
)

train_query_embeddings = embedder.encode(
    train_data["mcq_query"].to_list(), 
    show_progress_bar=True,
    batch_size=32
).astype(np.float32) # type: ignore

test_query_embeddings = embedder.encode(
    test_data["mcq_query"].to_list(), 
    show_progress_bar=True,
    batch_size=32
).astype(np.float32) # type: ignore

del embedder
reset_gpu()

In [ ]:
# Reduce high-dimensional embeddings to 10 dimensions using PaCMAP for clustering
pacmap = PaCMAP(
    n_components=10,
    n_neighbors=30,
    apply_pca=True,
    distance="euclidean",
    random_state=SEED
)

pacmap_data = pacmap.fit_transform(train_query_embeddings)

clusterer = HDBSCAN(
    min_cluster_size=5,
    min_samples=3,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

cluster_labels = clusterer.fit_predict(pacmap_data) # type: ignore

In [ ]:
train_retrieved_chunks = retrieve_top_k(train_query_embeddings, k=10)
train_data = train_data.with_columns(
    pl.Series("retrieved_chunks", train_retrieved_chunks),
    pl.Series("cluster", cluster_labels)
)
train_data.write_parquet(inter_path_config["train_retrieved"])

test_retrieved_chunks = retrieve_top_k(test_query_embeddings, k=10)
test_data = test_data.with_columns(
    pl.Series("retrieved_chunks", test_retrieved_chunks)
)
test_data.write_parquet(inter_path_config["test_retrieved"])

In [ ]:
%%writefile /kaggle/working/rerank_worker.py
import gc, sys, argparse

from tqdm.auto import tqdm

import numpy as np
import polars as pl

import torch

from sentence_transformers import CrossEncoder
from transformers import BitsAndBytesConfig

def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--input", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--model_path", required=True)
    p.add_argument("--instruction", required=True)
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--k", type=int, default=5)
    args = p.parse_args()

    payload = pl.read_parquet(args.input)
    query_texts = payload["query"].to_list()
    retrieved_chunks = payload["retrieved_chunks"].to_list()

    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=False
    )

    model = CrossEncoder(
        args.model_path,
        trust_remote_code=True,
        model_kwargs={
            "quantization_config": bnb_config,
            "attn_implementation": "sdpa",
            "dtype": torch.float16,
            "device_map": "cuda:0"
        },
        prompts={"rerank": args.instruction},
        default_prompt_name="rerank"
    )

    if model.tokenizer.pad_token is None:
        model.tokenizer.pad_token = model.tokenizer.eos_token
        
    model.model.config.pad_token_id = model.tokenizer.pad_token_id
    model.model.config.use_cache = False
    model.model.eval()

    flat_pairs, boundaries = [], []
    idx = 0

    for q, chunks in zip(query_texts, retrieved_chunks):
        chunks = [c if c.strip() else " " for c in chunks]
        flat_pairs.extend((q, c) for c in chunks)
        boundaries.append((idx, idx + len(chunks)))
        idx += len(chunks)

    flat_scores = model.predict(
        flat_pairs,
        batch_size=args.batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    top_chunks_per_query = []
    for (start, end), chunks in zip(boundaries, retrieved_chunks):
        scores = flat_scores[start:end]
        top_idx = np.argsort(-scores)[:args.k]
        top_chunks_per_query.append([chunks[i] for i in top_idx])

    pl.DataFrame({"top_chunks": top_chunks_per_query}).write_parquet(args.output)

if __name__ == "__main__":
    main()

In [ ]:
def rerank_split_gpu(
        query_texts: list[str], 
        retrieved_chunks: list[list[str]], 
        tag: str, 
        k: int = 5, 
        batch_size: int = 16
) -> list[list[str]]:

    mid = len(query_texts) // 2
    halves = {
        0: (query_texts[:mid], retrieved_chunks[:mid]),
        1: (query_texts[mid:], retrieved_chunks[mid:]),
    }

    input_paths, output_paths = {}, {}
    for gpu_id, (qs, chunks) in halves.items():
        input_paths[gpu_id] = f"/kaggle/working/{tag}_{gpu_id}_input.parquet"
        output_paths[gpu_id] = f"/kaggle/working/{tag}_{gpu_id}_output.parquet"

        pl.DataFrame({"query": qs, "retrieved_chunks": chunks}).write_parquet(input_paths[gpu_id])

    procs = []
    for gpu_id in (0, 1):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_id)}
        cmd = [
            sys.executable, "/kaggle/working/rerank_worker.py",
            "--input", input_paths[gpu_id],
            "--output", output_paths[gpu_id],
            "--model_path", model_config["reranker_model"],
            "--instruction", MCQ_RERANK_INSTRUCTION,
            "--batch_size", str(batch_size),
            "--k", str(k)
        ]
        procs.append(subprocess.Popen(cmd, env=env))

    for proc in procs:
        if proc.wait() != 0:
            raise RuntimeError("rerank_worker.py failed - check the cell output above for the traceback")

    top0 = pl.read_parquet(output_paths[0])["top_chunks"].to_list()
    top1 = pl.read_parquet(output_paths[1])["top_chunks"].to_list()
    
    return top0 + top1

In [ ]:
train_retrieved_chunks = pl.read_parquet(inter_path_config["train_retrieved"])["retrieved_chunks"].to_list()
train_ranked_chunks = rerank_split_gpu(
    train_data["mcq_query"].to_list(), 
    train_retrieved_chunks, 
    tag="train"
)
train_data = train_data.with_columns(pl.Series("ranked_chunks", train_ranked_chunks))
train_data.write_parquet(inter_path_config["train_ranked"])

test_retrieved_chunks = pl.read_parquet(inter_path_config["test_retrieved"])["retrieved_chunks"].to_list()
test_ranked_chunks = rerank_split_gpu(
    test_data["mcq_query"].to_list(), 
    test_retrieved_chunks, 
    tag="test"
)
test_data = test_data.with_columns(pl.Series("ranked_chunks", test_ranked_chunks))
test_data.write_parquet(inter_path_config["test_ranked"])

In [ ]:
# Delete intermediate files in disk
patterns = ["*_input.parquet", "*_output.parquet"]

for pattern in patterns:
    for file_path in inter_path_config["working_dir"].glob(pattern):
        if file_path.is_file():
            
            file_path.unlink()
            print(f"Successfully deleted: {file_path.name}")

In [ ]:
def average_precision_at_3(pred_letters: list[str], true_letter: str) -> float:
    for i, p in enumerate(pred_letters[:3]):
        if p == true_letter:
            return 1.0 / (i + 1)
    return 0.0

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_config["generative_slm"], 
    trust_remote_code=True,
    truncation_side="left",
    padding_side="left"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

OPTION_TOKEN_IDS = {
    opt: tokenizer(f" {opt}", add_special_tokens=False)["input_ids"][-1]
    for opt in OPTION_COLS
}
reset_gpu()

model = AutoModelForCausalLM.from_pretrained(
    model_config["generative_slm"],
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    trust_remote_code=True,
    dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False
model.eval()

In [ ]:
def build_generation_prompt(row: dict, chunks: list[str]) -> str:
    context = "\n\n".join(chunks)
    options_block = "\n".join(
        f"{opt}) {row[opt]}" for opt in OPTION_COLS if row[opt] is not None
    )
    
    messages = [
        {"role": "system", "content": GENERATIVE_SLM_INSTRUCTION},
        {"role": "user", "content": (
            f"Context:\n{context}\n\n"
            f"Question: {row['prompt']}\n"
            f"Options:\n{options_block}\n\n"
            "Answer:"
        )},
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
def rank_options(
    data: pl.DataFrame, 
    top_chunks: list[list[str]], 
    batch_size: int = 8
) -> list[str]:
    
    rows = data.to_dicts()
    predictions = []

    with torch.inference_mode():
        for start in tqdm(range(0, len(rows), batch_size), desc="Scoring"):
            batch_rows = rows[start:start + batch_size]
            batch_chunks = top_chunks[start:start + batch_size]

            prompts = [build_generation_prompt(r, c) for r, c in zip(batch_rows, batch_chunks)]
            enc = tokenizer(prompts, padding=True, truncation=True, return_tensors="pt").to(model.device)

            logits = model(**enc).logits[:, -1, :]

            for i, row in enumerate(batch_rows):
                valid_opts = [opt for opt in OPTION_COLS if row[opt] is not None]
                opt_logits = torch.tensor([logits[i, OPTION_TOKEN_IDS[opt]].item() for opt in valid_opts])
                ranked = [valid_opts[j] for j in torch.argsort(opt_logits, descending=True)]
                
                predictions.append(" ".join(ranked))

    return predictions

In [ ]:
def predict_top3(
    data: pl.DataFrame, 
    top_chunks: list[list[str]], 
    batch_size: int = 8
) -> list[str]:
    
    rows = data.to_dicts()
    predictions = []

    with torch.inference_mode():
        for start in tqdm(range(0, len(rows), batch_size), desc="Scoring"):
            batch_rows = rows[start:start + batch_size]
            batch_chunks = top_chunks[start:start + batch_size]

            prompts = [build_generation_prompt(r, c) for r, c in zip(batch_rows, batch_chunks)]
            enc = tokenizer(
                prompts, 
                padding=True, 
                truncation=True, 
                return_tensors="pt"
                ).to(model.device)

            logits = model(**enc).logits[:, -1, :]

            for i, row in enumerate(batch_rows):
                valid_opts = [opt for opt in OPTION_COLS if row[opt] is not None]
                opt_logits = torch.tensor([logits[i, OPTION_TOKEN_IDS[opt]].item() for opt in valid_opts])
                ranked = [valid_opts[j] for j in torch.argsort(opt_logits, descending=True)]
                
                predictions.append(" ".join(ranked[:3]))

    return predictions

In [ ]:
def run_full_pipeline_eval(
    data: pl.DataFrame, 
    chunks_list: list[list[str]],
    group_col: str, 
    batch_size: int = 4
) -> tuple[pl.DataFrame, pl.DataFrame]:
    
    true_letters = data["answer"].to_list()

    full_preds = rank_options(data, chunks_list, batch_size=batch_size)
    top3_preds = [" ".join(p.split()[:3]) for p in full_preds]
    ap3 = [average_precision_at_3(p.split(), t) for p, t in zip(top3_preds, true_letters)]

    result = data.select(group_col).with_columns(
        pl.Series("pred", top3_preds),
        pl.Series("ap3", ap3),
    )

    overall_map3 = result["ap3"].mean()
    print(f"Overall MAP@3 (full data, n = {len(result)}): {overall_map3:.8f}")

    group_stats = (
        result.group_by(group_col)
        .agg(pl.col("ap3").mean().alias("group_map3"), pl.len().alias("n"))
        .sort("group_map3")
    )
    print(f"\nWorst-performing groups:\n{group_stats.head(10)}")
    print(f"\nStd across groups: {group_stats['group_map3'].std():.4f} "
          f"(high = topic-dependent)")

    return result, group_stats

In [ ]:
train_result, train_group_stats = run_full_pipeline_eval(
    train_data, train_ranked_chunks, group_col="cluster", batch_size=4
)

In [ ]:
# train_ranked_chunks = pl.read_parquet(inter_path_config["train_ranked"])["ranked_chunks"].to_list()
# train_data = train_data.with_columns(pl.Series("top3_pred", predict_top3(train_ranked_chunks, train_ranked))) # type: ignore
# train_data.write_parquet(inter_path_config["train_final"])

# test_ranked_chunks = pl.read_parquet(inter_path_config["test_ranked"])["ranked_chunks"].to_list()
# test_data = test_data.with_columns(pl.Series("top3_pred", predict_top3(test_data, test_ranked_chunks)))

# test_retrieved_chunks = pl.read_parquet(inter_path_config["test_retrieved"])["retrieved_chunks"].to_list()
# test_data = test_data.with_columns(pl.Series("top3_pred", predict_top3(test_data, test_retrieved_chunks)))

# test_data.write_parquet(inter_path_config["test_final"])

In [ ]:
# submission = test_data.select(["id", "top3_pred"]).rename({"id": "ID", "top3_pred": "Prediction"})
# submission.write_csv(inter_path_config["submission"])